# Model validation

## How to check a model's goodness of fit?

Model validation starts with checking that the model captures the key patterns in the data, and that its parameters make a meaningful contribution to the model. It also includes checking that the assumptions behind the model are indeed met. This is all based on concepts that you have already seen in the previous chapters.

### Measures of fit

To assess whether a model fits the data properly, you can of course use the functions from which you got the best parameter estimates, i.e., the [sum of squares of the residuals](2:regression:residuals) or the [(log) likelihood](2:beyond_least_squares:likelihood). But since those metrics lack one bound, they are tricky to interpret. For instance, the lower bound of the sum of squares of the residuals is 0, which is the target (the model fits the data perfectly), but it has no upper bound. Then what is the maximum acceptable value for the the sum of squares of the residuals? This question is very much case and dataset specific, so it has no general answer.

To circumvent it, we turn to the [coefficient of determination](2:regression:determination), denoted $R^2$, which measures the proportion of the outcome's variation that is predictable by the model. Its main advantage is that it has clearly interpretable bounds: 1 implies that model explains all of the outcome's variation, 0 implies that it explains none of it.

```{tip}

To be more specific, a coefficient of determination of 0 implies that the model is no better than using the sample mean of the outcome as prediction, no matter the value of the predictor(s). Technically, the R<sup>2</sup> has a upper bound, 1, but no lower bound: in specific cases, it can be negative, which implies that the model makes worse predictions than just using the sample mean of the outcome.
```

The coefficient of determination was developed with linear regression, and has become a widely used metric when the outcome is a continuous variabe. But it is not valid with all models: it is not appropriate with logistic regression, where [McFadden's pseudo $R^2$](2:logistic_regression:pseudo) is often used instead. While getting values of $R^2$ above 0.8 is possible, and a sign of a good fit, the pseudo $R^2$ is generally much lower, and values above 0.2 already represent a good fit.

### Parameter and model significance

To assess whether the parameters of a model make a significant contribution to its predictions, you can check their standard errors and 95th percent confidence intervals: large standard errors imply uncertain parameter estimates, and a 95th confidence interval that includes 0 might imply an insignificant parameter (and predictor if that parameter is a slope). You can get standard errors and confidence intervals for the parameters of a linear model [by assuming a normally-distributed error](2:uncertainty:parameters). And you can get similar estimates for logistic regression [based on the Bernoulli distribution](2:logistic_regression:confidence). [Bootstrapping](2:uncertainty:bootstrap) can also be used, especially when the [assumptions behind a model](2:regression:assumptions) are not met.

You can also check significance of a linear model's parameters [using a two-sided $t$-test](2:uncertainty:test_params) with the null hypothesis that the parameter is equal to 0. If the null hypothesis can be rejected, then the parameter can be considered to make a significant contribution to the model's predictions. You can follow the [same procedure for logistic regression](2:logistic_regression:confidence), just with $z$-tests instead of $t$-tests. In the case of linear regression, you can also test the significance of the whole model using a $F$-test with the null hypothesis that all the model's parameters are equal to 0.

### Residual analysis

To assess whether the [assumptions behind ordinary least squares](2:regression:assumptions) are met, you can check that the residuals are pure random noise with a mean of 0, i.e., that they do not show any pattern or bias. This process is called [residual analysis](2:regression:residual_analysis), and starts with plotting the residuals against the predictor(s) or the predictions. Different hypothesis tests and statistics, like the Durbin-Watson statistic, also exist to assess the randomness of the residuals.

You can also use check [measures of shape](1:descriptive_statistics:shape) like the skewness and kurtosis to quantify how close they are from a normal distribution. [Hypothesis tests of normality](2:uncertainty:test_residuals) like the Jarque-Bera test can complete this analysis.

## How to check a model's predictive performance?

Only validating a model on the data used for fitting often leads to an overly optimistic assessment of the model's predictive performance. Ideally, we would like to acquire new data to test the model's ability to predict them, so to test the model's ability to generalize beyond the data used for fitting. But acquiring new data just for validation is rarely possible, so you need to turn to other strategies to estimate the error that the model would make on new data, called the generalization error.

### Metrics

The first step when validating predictive performance is to choose a metric to quantify generalization error. This is quite similar to what you have seen in optimization, where you have to define an [objective function](2:beyond_least_squares:objective). It is not uncommon to select multiple metrics to get a different perspective, but some metrics are limited to a specific variable type.

#### For continuous outcomes

Metrics used with models predicting continuous variables, like linear regression, are similar to measures you have already seen:

  * The **mean squared error** directly measures the error on the new data, giving more weight to the highest errors compared to the lowest ones:

    $$
        MSE = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2
    $$
    
    Where:
      * $n$ is the number of new data.
      * $y_i$ is a value of the outcome from the new data.
      * $\hat{y}_i$ is the prediction of the outcome by the model from the value(s) of the predictor(s) from the new data.

    Applying a square root to the $MSE$ leads to the root mean squared error ($RMSE$), which has the advantage of being in the same unit as the outcome, so being more easily interpretable.

  * The **mean absolute error** directly measures the error on the new data as well, giving equal weight to all the errors:

    $$
        MAE = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|
    $$
    
    It is also in the same unit as the outcome.

  * The **coefficient of determination** measures the proportion of the outcome's variation that is predictable by the model, as mentioned in the previous subsection:

    $$
        R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{\sum_{i=1}^n (y_i - \overline{y})^2}
    $$
    
    Where $\overline{y}$ is the mean of the outcome in the new data. It has the advantage of being easily interpretable, but the drawback that the mean might not be representative if the set of new data is too small. 

  <!-- * The log likelihood is also a valid metric, although it is less often used because it requires an estimate of the variance of the error, which is not available with all models. But when such estimate is available, like with linear regression, it allows you to consider error and uncertainty with one metric: the goal is to get a low error and a low uncertainty. -->

#### For categorical outomes

Classification, like logistic regression, requires a different set of metrics meant for categorical variables. All those metrics start from the confusion matrix, which counts the number of correct and incorrect classifications (here for binary classification, i.e., two classes, more classes would lead to a bigger matrix):

|                     | Predicted positive | Predicted negative  |
| :-----------------: | :----------------: | :-----------------: |
| **Actual positive** | True positive      | False negative      |
| **Actual negative** | False positive     | True negative       |

This matrix highlights the similarities between classification and hypothesis testing (and its [type I and II errors](2:hypothesistesting:significance)). It also provides an overview of how well the model predicts the different classes. Several metrics are derived from it:

  * The **accuracy** measures the proportion of correct classifications:

    $$
        \text{accuracy} = \frac{\text{correct classifications}}{\text{all classifications}} = \frac{\text{true positives} + \text{true negatives}}{\text{true positives} + \text{true negatives} + \text{false positives} + \text{false negatives}}
    $$

    which is easy to interpret and explain but sensitive to class imbalance (i.e., when a category has more data than another).

  * The **recall**, also called the true positive rate, measures the proportion of actual positives correctly classified:

    $$
        \text{recall} = \frac{\text{correctly classified actual positives}}{\text{all actual positives}} = \frac{\text{true positives}}{\text{true positives} + \text{false negatives}}
    $$

  * The **precision** measures the proportion of correct positive predictions:

    $$
        \text{precision} = \frac{\text{correctly classified actual positives}}{\text{all classified as positives}} = \frac{\text{true positives}}{\text{true positives} + \text{false positives}}
    $$

  * The **F1 score** is the harmonic mean of precision and recall:

    $$
        F_1 = \frac{2}{\displaystyle\frac{1}{\text{precision}} + \frac{1}{\text{recall}}} = 2\frac{\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}} = \frac{2\cdot\text{true positives}}{2\cdot\text{true positives} + \text{false positives} + \text{false negatives}}
    $$

    It lies between 0 and 1, and is robust to class imbalance.

  <!-- * The **ROC-AUC** measures the area under the ROC curve. ROC stands for *Receiver Operating Characteristic*, which is linked to its origin in radar detection. The ROC curve plots the false positive rate against the true positive rate for different threshold values (remember that logistic regression predicts a probability, which needs to be thesholded to get classes; this threshold is often 0.5 by default, but it can be changed). It lies between 0 and 1, with 1 being a perfect classifier. A random classifier would return 0.5, so any value below that means poor performances. It is also robust to class imbalance. -->

### Cross-validation

Cross-validation is an ensemble of approaches that use resampling to mimic the process of obtaining new data and checking a model's predictive performance on them. The idea is to split the data into two subsets:
  * A training set that will be used to fit the model.
  * A test set that is not used at all for fitting, and acts as new data to validate the model based on the selected metric(s).

Different strategies exist to get those subsets:

  * The simplest strategy, called **holdout validation**, consists in randomly selecting some observations to be in the test set, usually between 10 and 30% of the data. It assumes that the data are [independent and identically distributed](1:clt:iid). While this approach is simple and efficient, it only works well with large datasets (with tens of thousands of observations or more). With small datasets, the risk is to end up with a biased test set containing observations that the model can predict either very well or very poorly, so to be overly optimistic or overly pessimistic about the model's performance.

  * A more robust strategy, called **$\boldsymbol{k}$-fold cross-validation**, consists in randomy splitting the data into several subsets of equal size called folds, usually 5 or 10 folds. Each fold is used once as test set, while the others are used to fit the model. A metric value is computed for each set, and the resulting values are averaged to get a single value for the model. It also assumes that the data are [independent and identically distributed](1:clt:iid). The advantage of $k$-fold cross-validation is that all the data are tested (and tested exactly once).

  * A deterministic strategy, called **Leave-one-out cross-validation**, consists in testing all the data individually, which is equivalent to $k$-fold cross-validation with $n$ folds for a dataset of $n$ observations. It has the advantage of removing any bias due to randomly selecting folds. But, since it only removes one datum at a time, the training sets remain quite similar, and the models as well, which can lead to misestimate the generalization error. $k$-fold cross-validation with the usual 5 or 10 folds is more robust from that perspective, on top of being less expensive.

  * Another deterministic strategy, called **Leave-one-group-out cross-validation**, consists in using predefined groups within the data as folds. Those groups can be different entities like different wells, different catchments, different countries, etc. The goal is then to assess how a model fitted with data of some specific groups performs on new, previously unseen groups. It has the advantage of relaxing the assumption that the data are [independent and identically distributed](1:clt:iid), because the data within each group can be dependent.

One thing to be particularly mindful of in cross-validation is data leakage. Data leakage happens when information is shared between training and test sets, and leads to an overly optimistic assessment of model performance. It can happen when some pre-processing steps like [standardization](2:data_wrangling:standardization), [imputation of missing data](2:data_wrangling:missing_data), or [outlier removal](2:data_wrangling:outliers) are performed from the entire dataset and not from the training set only. It can also happen when the chosen cross-validation strategy does not match how the model will be applied in practice. For instance, if the model is meant to predict a new group, e.g., it will be applied to catchments it was not fitted on, then $k$-fold cross-validation will lead to an overly optimistic assessment, while leave-one-group-out cross-validation will be far less biased.

(2:validation:activity)=
## Activity: Cross-validating discharge predictions

Now let's have a look at assessing a linear model's predictive performance using cross-validation. For that, make sure to start the interactive Python environment by clicking on {fa}`rocket` {fa}`arrow-right-long` {guilabel}`Live Code` at the top of this page (then wait until the Python interaction is ready).

First, you need to import some packages:

In [ ]:
# To find the right path to the data
from pathlib import Path
# To manipulate arrays and matrices
import numpy as np
# To manipulate and analyze data
import pandas as pd
# To build design matrices and outcome vectors
from patsy import dmatrices
# To create statistical models
import statsmodels.api as sm
# To perform k-fold cross-validation
from sklearn.model_selection import KFold
# To compute the coefficient of determination
from sklearn.metrics import r2_score

Then, load the data into a dataframe (data from [10.5066/P9SHOOH0](https://www.sciencebase.gov/catalog/item/631405eed34e36012efa3505)):

In [ ]:
data = pd.read_csv(Path.cwd().parent/'../data/ohio_streams_usgs_2005_5153.csv')
data.head()

The goal is to check the predictive performance of a linear model predicting discharge from river width.

Let's start by looking at the goodness of fit of the model. First, you need to define the model and fit it to the data:

In [ ]:
y, X = dmatrices('bankfull_discharge ~ bankfull_width', data=data)
model = sm.OLS(y, X)
results = model.fit()

Then you can predict the outcome for the data:

In [ ]:
y_pred = results.predict(X)

Finally, you can compute the coefficient of determination on all the data using the function `r2_score`:

In [ ]:
r2_score(y, y_pred)

Rather than implementing cross-validation from scratch, you will use [scikit-learn](https://scikit-learn.org/stable/), a Python package for machine learning. Let's start by getting familiar with $k$-fold cross-validation in scikit-learn.

First, you need to define a $k$-fold cross-validator using [scikit-learn's class `KFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html):

In [ ]:
cv = KFold(n_splits=2, shuffle=True, random_state=42)

You can see that `KFold` takes three parameters:
  * `n_splits`, the number of folds.
  * `shuffle`, to randomly shuffle the data or not before splitting the folds.
  * `random_state`, the seed to make the results reproducible when `shuffle` is `True`.

Then, you can use [the function `split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html#sklearn.model_selection.KFold.split), which takes as input the data, to get the indices of the training and test sets for each fold:

In [ ]:
for i, (train_index, test_index) in enumerate(cv.split(data)):
    print(f'Fold {i}:')
    print(f'   Train: index={train_index}')
    print(f'   Test:  index={test_index}')

Check what happens when you change the number of folds, when you use shuffling or not, and when you change the random state.

Now quantify the predictive performance of the linear model using $k$-fold cross-validation with 5 folds and the coefficient of determination as metric. For each fold, you need to:
  * Get the training and test data using the indices given by the function `split`; for that you need to use pandas' `loc`, e.g., to get the training data: `data.loc[train_index]`.
  * Fit a simple linear model on the training set.
  * Compute the coefficient of determination on the test set.

That will give you 5 coefficients of determination, and the mean coefficient of determination will give you the final estimate of the generalization error.

In [ ]:
# Your answer here

How does the mean coefficient of determination from $k$-fold cross-validation compare with the coefficient of determination computed on all the data?